# ADW Mean-Field Gap Equation: Technical Notebook

**Phase 3 Wave 3** — Tests the gravity wall via the Akama-Diakonov-Wetterich mechanism.

This notebook reproduces the key calculations from Paper 5:
1. Wen's emergent QED (N_f = 4 Dirac fermions from string-net condensation)
2. Hubbard-Stratonovich decomposition of the 8-fermion interaction
3. Coleman-Weinberg effective potential and gap equation
4. Phase diagram: pre-geometric → vestigial metric → full tetrad
5. Fluctuation analysis: Vergeles mode counting and graviton identification
6. Structural obstacles for the emergent fermion bootstrap

In [ ]:
import numpy as np
from src.core.visualizations import COLORS, apply_layout
from src.adw.wen_model import wen_coulomb_phase, herbut_terminal_velocity
from src.adw.hubbard_stratonovich import TetradField, adw_cosmological_term, hubbard_stratonovich_decompose
from src.adw.gap_equation import (
    full_gap_analysis, critical_coupling, effective_potential_landscape,
    GapEquationParams, PhaseType, vestigial_metric_condition,
)
from src.adw.fluctuations import (
    full_fluctuation_analysis, vergeles_mode_counting,
    graviton_identification, structural_obstacles,
)
from src.core.formulas import adw_critical_coupling, graviton_polarization_count

## 1. Wen's Emergent QED

The Levin-Wen rotor model on a 3D cubic lattice produces emergent QED_{3+1} in the Coulomb phase. The Nielsen-Ninomiya theorem mandates 2^3 = 8 Weyl species (4 Dirac fermions). These are non-chiral.

In [ ]:
qed = wen_coulomb_phase()
print(f"Emergent theory: {qed.gauge_group} with N_f = {qed.N_f} Dirac fermions")
print(f"Weyl species: {qed.N_weyl} (from 2^3 = 8 BZ corners)")
print(f"Chiral: {qed.is_chiral}")

### Herbut Velocity Equalization

RG flow drives v_F, v_B, c toward a common terminal velocity, restoring emergent Lorentz invariance.

In [ ]:
v_initial = qed.velocities
v_final = herbut_terminal_velocity(v_initial.v_F, v_initial.v_B, v_initial.c_lattice, rg_steps=500)
print(f"Initial Lorentz violation: {v_initial.lorentz_violation:.4f}")
print(f"Final Lorentz violation:   {v_final.lorentz_violation:.4f}")

## 2. ADW Interaction and HS Decomposition

The 8-fermion cosmological term requires a double Hubbard-Stratonovich transformation. After decomposition, the fermion action becomes the Dirac action in curved spacetime.

In [ ]:
action = adw_cosmological_term(G=1.0)
hs = hubbard_stratonovich_decompose(action)
print(f"Interaction: {action.interaction_order}")
print(f"HS steps needed: {hs.n_hs_steps}")
print(f"Fermion quadratic after HS: {hs.is_fermion_quadratic}")
print(f"Yukawa: {hs.yukawa_structure}")

## 3. Effective Potential and Critical Coupling

The Coleman-Weinberg potential V_eff(C) = C^2/(2G) - (N_f/16pi^2)[Lambda^2 C^2 - C^4 ln(Lambda^2/C^2 + 1)] develops a nontrivial minimum for G > G_c.

In [ ]:
# viz-ref: fig_adw_effective_potential
from src.core.visualizations import fig_adw_effective_potential
fig_adw_effective_potential().show()

In [ ]:
Lambda, N_f = 1.0, qed.N_f
G_c = critical_coupling(Lambda, N_f)
print(f"Critical coupling G_c = 8pi^2/(N_f * Lambda^2) = {G_c:.4f}")
print(f"Consistency check: formulas.py gives {adw_critical_coupling(Lambda, N_f):.4f}")

## 4. Gap Equation Solution

For G > G_c, the gap equation admits a nontrivial solution with Lorentzian signature.

In [ ]:
for ratio in [0.5, 1.0, 1.5, 2.0, 5.0]:
    result = full_gap_analysis(G=ratio * G_c, Lambda=Lambda, N_f=N_f)
    phase = result.phase.value
    C = result.nontrivial_solution.C if result.nontrivial_solution else 0.0
    lor = result.nontrivial_solution.is_lorentzian if result.nontrivial_solution else '-'
    print(f"G/G_c = {ratio:.1f}: phase = {phase:20s}  C = {C:.4f}  Lorentzian = {lor}")

### Phase Diagram

In [ ]:
# viz-ref: fig_adw_phase_diagram
from src.core.visualizations import fig_adw_phase_diagram
fig_adw_phase_diagram().show()

### Tetrad Properties at the Minimum

The flat-spacetime ansatz e^a_mu = C delta^a_mu automatically gives Lorentzian metric g_{mu nu} = C^2 eta_{mu nu}.

In [ ]:
result = full_gap_analysis(G=2 * G_c, Lambda=Lambda, N_f=N_f)
sol = result.nontrivial_solution
print(f"Tetrad VEV: C = {sol.C:.6f}")
print(f"Metric signature: {sol.tetrad.signature}")
print(f"Lorentzian: {sol.is_lorentzian}")
print(f"det(e) = {sol.tetrad.determinant:.6f}")
print(f"V_eff(C_min) = {sol.V_eff:.8f}")

## 5. Fluctuation Analysis

The symmetry breaking L_c x L_s -> L_J produces gravitons as Higgs bosons of the tetrad order parameter.

In [ ]:
fluct = full_fluctuation_analysis()
print(f"SSB: {fluct.ssb_pattern.group_name} -> {fluct.ssb_pattern.subgroup_name}")
print(f"Broken generators: {fluct.ssb_pattern.n_broken}")
print()
for m in fluct.modes:
    print(f"  {m.mode_type.value:20s}: {m.count} modes ({m.mass_status})")
print(f"\nVergeles check: {fluct.vergeles_check}")

In [ ]:
# viz-ref: fig_adw_ng_mode_decomposition
from src.core.visualizations import fig_adw_ng_mode_decomposition
fig_adw_ng_mode_decomposition().show()

### Graviton Identification

The graviton is a **Higgs boson** (not a Nambu-Goldstone boson) of the tetrad order parameter. The NG modes are absorbed by the spin connection.

In [ ]:
grav = graviton_identification(4)
print(f"Polarizations: {grav['n_polarizations']}")
print(f"Spin: {grav['spin']}")
print(f"Mass: {grav['mass']}")
print(f"Nature: {grav['nature']}")
print(f"Gravitational waves: {grav['has_gravitational_waves']}")

## 6. Structural Obstacles

Four obstacles prevent the emergent fermion bootstrap at Level 2.

In [ ]:
# viz-ref: fig_adw_structural_obstacles
from src.core.visualizations import fig_adw_structural_obstacles
fig_adw_structural_obstacles().show()

## 7. Coupling Scan Across N_f

In [ ]:
# viz-ref: fig_adw_coupling_scan
from src.core.visualizations import fig_adw_coupling_scan
fig_adw_coupling_scan().show()

## Summary

| Result | Value |
|--------|-------|
| Critical coupling | G_c = 8pi^2/(N_f Lambda^2) |
| Nontrivial solution | Yes (G > G_c) |
| Lorentzian signature | Automatic for flat ansatz |
| Graviton polarizations | 2 (massless spin-2) |
| Vergeles mode counting | 16 = 6 + 4 + 2 + 4 |
| Structural obstacles | 4 (all open) |
| Lean theorems | 21 (1 sorry for Aristotle) |